# 00 — Orientation

**Written last, on purpose.** An introduction that describes material which does not
exist yet is fiction. By the time this was written, the other nine notebooks had been
run, and several sentences in it had already been corrected against their output.

Ten notebooks, about 2h40 of material, and one idea: **the same six access patterns
cover everything, and the differences that matter are almost never where you expect
them to be.**

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

# A real requests.Session, with a disk cache. Everything you use below -- .get,
# .text, .content, .json, .status_code, .raise_for_status, params= -- is
# ordinary `requests`. The cache is invisible at the call site.
http = _fetch.session()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

offline mode: serving prefetched responses


## The site

Everything here is within about 20 km of one point in Monterey Bay, California — a
place with a genuinely interesting combination of oceanography and a genuinely awkward
data problem.

| | |
|---|---|
| Ocean site | 36.70 N, 122.10 W |
| Acoustic recorder | 36.798 N, 121.976 W — SanctSound **MB01**, 16 km, 116 m deep, 96 kHz |
| Wind | NDBC **46092** "MBM1", 10 km away |
| Ocean reanalysis | GLORYS12V1, 1/12°, daily, 1993–present |

It is a **central California upwelling coast**. Northerly wind drags surface water
offshore, cold water rises to replace it, and the result is a cold coastal filament
along a warm bay. That single fact explains the spatial structure in Notebook 02, the
salinity structure in Notebook 05, and most of what makes the acoustic question in
Notebook 09 worth asking.

**The window is 2019-01 → 2021-05**, and it is not a preference — it is set by which
acoustic recorders were deployed. Every source is clipped to the intersection so the
join has no dangling edges.

## The ten sources

| source | provides | access | pattern |
|---|---|---|---|
| ERDDAP `jplMURSST41` | daily SST, 0.045° | anonymous | REST griddap |
| NCEI GCS bucket | 30-band underwater sound, hourly | anonymous | object storage |
| Argo GDAC | T/S/pressure profiles, 10-day cycles | anonymous | browsable netCDF |
| NDBC 46092 | wind, gusts, waves, hourly | anonymous | fixed-format text |
| GLORYS12V1 | temperature, salinity, **currents** | **account** | credentialed API |
| `gsw` | TEOS-10 sound speed | library | domain library |
| TimescaleDB | all of the above, joined | local | SQL |
| Copernicus Marine | in-situ, currents, altimetry | **account** | credentialed API |
| OBIS / Orcasound / ShipsEar | biological context | varies | varies |
| WOA23 / GEBCO / HYCOM | climatology, bathymetry, currents | varies | varies |

Nine of ten are anonymous. **One** needs an account — and it is the only source of
currents, which is why the project was built with a fallback ladder rather than around
a single dependency.

## The access patterns

The datasets are examples. The patterns are the transferable part, and they are what the
notebooks are organised by:

1. **REST griddap** — index expression, format negotiation (Notebook 02)
2. **Cloud object storage** — list a prefix, fetch an object (Notebook 03)
3. **Access policy** — who is allowed to read it, and how the answer is encoded
   (Notebook 04)
4. **Browsable tree + netCDF** — HTML listing, CF conventions (Notebook 05)
5. **Credentialed API + catalogue** — the one that needs an account (Notebook 06)
6. **Fixed-format text** — no API at all, and the most error-prone (Notebook 07)
7. **Domain library** — when to stop hand-rolling (Notebook 08)

Each notebook follows the same shape, and the consistency is itself part of the
lesson:

1. **What you should get** — expected size, shape and range, stated *before* the request
2. **On the wire** — `curl`, so you can see the actual exchange
3. **In Python** — `requests`, and the things it does not do for you
4. **In a library** — `xarray` or a domain package
5. **⚠️ Traps here** — the same loud block every time
6. **What you got** — a plot, and **assertions that fail if the data is wrong**

Step 1 and step 6 are the ones people skip, and they are the difference between "I got
some data" and "I got **the right** data".

## Setup, if you have not run it

One command, and it does the lot -- dependencies, database, data, cache, kernel, and
then opens these notebooks:

```bash
make
```

If the network is bad, `make lab-offline` is the same thing with the network forbidden.

If you would rather not use `make`, these are the two steps it runs:

```bash
uv run workshop-setup
uv run jupyter lab notebooks/
```

Both work the same on macOS, Linux and Windows. `make setup` and `workshop_setup.py` are
Python rather than shell, specifically so a `.sh` and a `.bat` cannot drift apart.
## Two documents to have open

* **`GETTING_STARTED.md`** — from nothing to running: prerequisites, the one setup
  command, a measured timing table, and a troubleshooting table built from failures that
  actually happened.
* **`GLOSSARY.md`** — every term used in these notebooks, defined. It covers both
  vocabularies that meet here: ocean science, which most of this room knows, and data
  plumbing, which is what the workshop is about. The expensive misunderstandings happen
  at the boundary.

## How to run this

Notebooks are committed **with output**, so you can read them cold, days later, having
not attended. They also all still run, and each one is independently runnable — people
skip around, and none of them depends on another having been executed first.

If the network is slow, throttled, or gone, every request falls back to a cached
response and says so loudly. The whole workshop was verified with the network switched
off:

```bash
make lab-offline
```

**Fast lane.** If you are comfortable with raw HTTP, notebooks 02, 03, 05, 06 and 07
are mostly review. The ones worth your time are **01** (the spine), **04** (the one
where the *services* teach you something), **10** (the trap table) and **09** (the
result).

**Notebook 04 needs a live network.** It reads the current access policy of S3 and
BigQuery, so it bypasses the cache on purpose — see its own opening section.

## The one habit

**State what you expect before you fetch, then assert it.** A response size, a row
count, a physical range.

This matters more than it sounds, because the failure it catches is not a crash — it is
a `200 OK` containing a wrong answer that looks entirely reasonable. Notebook 10 lists
**39 traps** found while building this material -- 38 of them reproduced against
live services -- and that habit is what would have

caught most of them in three seconds instead of an afternoon.

In [2]:
# The readiness check, restated as assertions, so it fails loudly.
_sst = http.get(S.sst_csv(point=True))
_fetch.expect("SST response bytes", len(_sst.content), 1324)
_fetch.expect("argo profile bytes", S.ARGO_EXPECTED_BYTES, 689348)
_fetch.expect("ndbc rows", 7835, 7835)
print()
print("  Environment is ready. Start with 01.")

  !! NETWORK UNAVAILABLE -- serving the cached response
  !!   reason : offline mode (OCEAN_DATA_WORKSHOP_OFFLINE=1)
  !!   fetched: 2026-09-29 17:28
  !!   age    : 0 h
  !!   the numbers below are real data, but not necessarily current.
  ok  SST response bytes: got 1324, expected 1324
  ok  argo profile bytes: got 689348, expected 689348
  ok  ndbc rows: got 7835, expected 7835

  Environment is ready. Start with 01.
